##Analytical Derivation of the Linear Kalman Filter

### Analytical Derivation of the Linear Kalman Filter

**Given System Model:**
$$x_k^- = A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}$$
$$y_k^- = H_kx_k^- + z_k$$

**Assumptions & Initial Conditions:**
* Prior state distribution: $x_{k-1}^+ \sim \mathcal{N}(m_{k-1}, P_{k-1})$
* Process noise: $w_{k-1} \sim \mathcal{N}(0, \Sigma_p)$
* Measurement noise: $z_k \sim \mathcal{N}(0, \Sigma_m)$
* The variables $x_{k-1}^+$, $w_{k-1}$, and $z_k$ are mutually independent.
* Linear combinations of Gaussian random variables are also Gaussian.

---

#### 1. Show that $x_k^- \sim \mathcal{N}(m_k^-, P_k^-)$

To find the distribution of the predicted state $x_k^-$, we calculate its expected value and variance.

**Expected Value (Mean):**
Apply the linear expectation operator $E[\cdot]$ to the state equation:
$$m_k^- \triangleq E[x_k^-] = E[A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}]$$
$$m_k^- = A_{k-1}E[x_{k-1}^+] + G_{k-1}E[w_{k-1}]$$
Since $E[x_{k-1}^+] = m_{k-1}$ and the noise has zero mean ($E[w_{k-1}] = 0$):
$$m_k^- = A_{k-1}m_{k-1}$$

**Variance (Covariance Matrix):**
Apply the variance operator. Since $x_{k-1}^+$ and $w_{k-1}$ are independent, the variance of their sum is the sum of their individual variances:
$$P_k^- \triangleq Var(x_k^-) = Var(A_{k-1}x_{k-1}^+) + Var(G_{k-1}w_{k-1})$$
Using the matrix variance property $Var(MX) = M Var(X) M^T$:
$$P_k^- = A_{k-1}Var(x_{k-1}^+)A_{k-1}^T + G_{k-1}Var(w_{k-1})G_{k-1}^T$$
$$P_k^- = A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T$$

Because the transformation is linear and the inputs are Gaussian, the output is Gaussian:
$$x_k^- \sim \mathcal{N}(m_k^-, P_k^-)$$

---

#### 2. Show that $y_k^- \sim \mathcal{N}(H_km_k^-, H_kP_k^-H_k^T + \Sigma_m)$

We repeat the process for the predicted measurement $y_k^-$.

**Expected Value:**
$$E[y_k^-] = E[H_kx_k^- + z_k] = H_kE[x_k^-] + E[z_k]$$
Substituting $E[x_k^-] = m_k^-$ and $E[z_k] = 0$:
$$E[y_k^-] = H_km_k^-$$

**Variance:**
Because $x_k^-$ depends only on prior states and process noise, it is strictly independent of the current measurement noise $z_k$:
$$Var(y_k^-) = Var(H_kx_k^- + z_k) = Var(H_kx_k^-) + Var(z_k)$$
$$Var(y_k^-) = H_kVar(x_k^-)H_k^T + Var(z_k)$$
Substituting $Var(x_k^-) = P_k^-$ and $Var(z_k) = \Sigma_m$:
$$Var(y_k^-) = H_kP_k^-H_k^T + \Sigma_m$$

Thus, the predicted measurement distribution is:
$$y_k^- \sim \mathcal{N}(H_km_k^-, H_kP_k^-H_k^T + \Sigma_m)$$

---

#### 3. Show the Joint Prior Distribution $[x_k^-, y_k^-]^T$

To define the joint multivariate Gaussian, we need the individual means, individual variances, and the cross-covariance between $x_k^-$ and $y_k^-$.

**Cross-Covariance Computation:**
$$Cov(x_k^-, y_k^-) = E[(x_k^- - E[x_k^-])(y_k^- - E[y_k^-])^T]$$
Substitute $y_k^- = H_kx_k^- + z_k$ and the expected values:
$$y_k^- - E[y_k^-] = (H_kx_k^- + z_k) - H_km_k^- = H_k(x_k^- - m_k^-) + z_k$$
Plug this back into the covariance equation:
$$Cov(x_k^-, y_k^-) = E\left[(x_k^- - m_k^-)\big(H_k(x_k^- - m_k^-) + z_k\big)^T\right]$$
Expand the transpose and distribute:
$$Cov(x_k^-, y_k^-) = E\left[(x_k^- - m_k^-)(x_k^- - m_k^-)^TH_k^T + (x_k^- - m_k^-)z_k^T\right]$$
Since $x_k^-$ and $z_k$ are independent and $z_k$ has zero mean, the expected value of the cross term is zero ($E[(x_k^- - m_k^-)z_k^T] = 0$).
$$Cov(x_k^-, y_k^-) = E[(x_k^- - m_k^-)(x_k^- - m_k^-)^T]H_k^T$$
Recognizing the definition of $P_k^-$:
$$Cov(x_k^-, y_k^-) = P_k^-H_k^T$$

The covariance matrix must be symmetric, so the bottom-left block is the transpose: $Cov(y_k^-, x_k^-) = H_kP_k^-$. Assembling the block matrix yields the joint distribution:

$$
\begin{bmatrix} x_k^- \\ y_k^- \end{bmatrix} \sim \mathcal{N} \left(
\begin{bmatrix} m_k^- \\ H_km_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^-H_k^T \\ H_kP_k^- & H_kP_k^-H_k^T + \Sigma_m \end{bmatrix}
\right)
$$

---

#### 4. Show the Updated Posterior Distribution $x_k^+$

When the measurement $y_k^{obs}$ is observed, we update our belief using the properties of conditional multivariate Gaussians.

**Theorem for Conditional Gaussians:**
If $\begin{bmatrix} X_1 \\ X_2 \end{bmatrix} \sim \mathcal{N}\left(\begin{bmatrix} \mu_1 \\ \mu_2 \end{bmatrix}, \begin{bmatrix} \Sigma_{11} & \Sigma_{12} \\ \Sigma_{21} & \Sigma_{22} \end{bmatrix}\right)$, then the conditional distribution $X_1 \mid X_2=a$ is Gaussian with:
* $\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(a - \mu_2)$
* $\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$

**Mapping our variables:**
* $X_1 = x_k^-$, and $X_2 = y_k^-$
* Observation $a = y_k^{obs}$
* $\mu_1 = m_k^-$ and $\mu_2 = H_km_k^-$
* $\Sigma_{11} = P_k^-$
* $\Sigma_{12} = P_k^-H_k^T$
* $\Sigma_{22} = H_kP_k^-H_k^T + \Sigma_m$

**Defining the Kalman Gain ($K_k$):**
We define the Kalman Gain as the matrix term $\Sigma_{12}\Sigma_{22}^{-1}$:
$$K_k \triangleq P_k^-H_k^T(H_kP_k^-H_k^T + \Sigma_m)^{-1}$$

**Calculating the Updated Mean ($m_k$):**
$$m_k \triangleq \mu_{1|2} = \mu_1 + K_k(a - \mu_2)$$
$$m_k = m_k^- + K_k(y_k^{obs} - H_km_k^-)$$

**Calculating the Updated Covariance ($P_k$):**
$$P_k \triangleq \Sigma_{1|2} = \Sigma_{11} - K_k\Sigma_{21}$$
$$P_k = P_k^- - K_k(H_kP_k^-)$$
Factor out $P_k^-$ on the right side:
$$P_k = (I - K_kH_k)P_k^-$$

Because the conditional slice of a Gaussian is strictly Gaussian, the updated state is:
$$x_k^+ \triangleq (x_k^- \mid y_k^- = y_k^{obs}) \sim \mathcal{N}(m_k, P_k)$$

---

#### 5. Find $E[x_k^- \mid y_k^- = y_k^{obs}]$ and $Var(x_k^- \mid y_k^- = y_k^{obs})$

By definition of the conditional distribution derived in step 4, these terms are exactly the updated posterior mean ($m_k$) and the updated posterior covariance ($P_k$).

**Conditional Expected Value:**
$$E[x_k^- \mid y_k^- = y_k^{obs}] = m_k = m_k^- + K_k(y_k^{obs} - H_km_k^-)$$

**Conditional Variance:**
$$Var(x_k^- \mid y_k^- = y_k^{obs}) = P_k = (I - K_kH_k)P_k^-$$

## 1-D Scalar Linear-Gaussian Kalman Filter Derivations

### 1-D Scalar Linear-Gaussian Kalman Filter Derivations

**Given 1-D System Model:**
$$x_k^- = a x_{k-1}^+ + w_{k-1}, \quad w_{k-1} \sim \mathcal{N}(0, q)$$
$$y_k^- = h x_k^- + z_k, \quad z_k \sim \mathcal{N}(0, r)$$

**Assumptions:**
* Prior state: $x_{k-1}^+ \sim \mathcal{N}(m_{k-1}, P_{k-1})$
* The variables $x_{k-1}^+$, $w_{k-1}$, and $z_k$ are mutually independent.

---

#### 1. Show the Prediction Step: $m_k^-$ and $P_k^-$

To find the distribution of the predicted state $x_k^-$, we calculate its expected value and variance using 1-D scalar properties.

**Expected Value (Predicted Mean $m_k^-$):**
Apply the linear expectation operator $E[\cdot]$:
$$m_k^- \triangleq E[x_k^-] = E[a x_{k-1}^+ + w_{k-1}]$$
$$m_k^- = a E[x_{k-1}^+] + E[w_{k-1}]$$
Since $E[x_{k-1}^+] = m_{k-1}$ and the process noise has zero mean ($E[w_{k-1}] = 0$):
$$\mathbf{m_k^- = a m_{k-1}}$$

**Variance (Predicted Covariance $P_k^-$):**
Apply the variance operator. Since $x_{k-1}^+$ and $w_{k-1}$ are independent:
$$P_k^- \triangleq Var(x_k^-) = Var(a x_{k-1}^+ + w_{k-1})$$
$$P_k^- = Var(a x_{k-1}^+) + Var(w_{k-1})$$
Using the scalar variance property $Var(cX) = c^2 Var(X)$:
$$P_k^- = a^2 Var(x_{k-1}^+) + Var(w_{k-1})$$
$$\mathbf{P_k^- = a^2 P_{k-1} + q}$$

---

#### 2. Show the Update Step: $m_k$ and $P_k$

We observe measurement $y_k^{obs}$. We must update our prior belief $x_k^- \sim \mathcal{N}(m_k^-, P_k^-)$ using Bayes' rule / conditioning.

First, define the innovation (measurement residual) $v_k$ and its variance $S_k$:
* **Innovation:** $v_k = y_k^{obs} - h m_k^-$
* **Innovation Variance:** $S_k = Var(h x_k^- + z_k) = h^2 P_k^- + r$

The **1-D Kalman Gain** is the ratio of the cross-covariance to the innovation variance:
$$K_k = \frac{Cov(x_k^-, y_k^-)}{Var(y_k^-)} = \frac{P_k^- h}{S_k}$$

**Updated Mean ($m_k$):**
Using the conditional Gaussian mean formula:
$$m_k = m_k^- + K_k v_k$$
Substitute the definition of $v_k$:
$$\mathbf{m_k = m_k^- + \frac{P_k^- h}{S_k} (y_k^{obs} - h m_k^-)}$$

**Updated Variance ($P_k$):**
Using the conditional Gaussian variance formula:
$$P_k = P_k^- - K_k Cov(y_k^-, x_k^-)$$
$$P_k = P_k^- - K_k (h P_k^-)$$
Factor out $P_k^-$:
$$\mathbf{P_k = (1 - K_k h) P_k^-}$$
Substitute $K_k = P_k^- h / S_k$ to show the alternative form:
$$P_k = \left(1 - \frac{P_k^- h}{S_k} h\right) P_k^-$$
$$\mathbf{P_k = \left(1 - \frac{P_k^- h^2}{S_k}\right) P_k^-}$$

---

#### 3. Show the Predictive Measurement Distribution $p(y_k^- \mid Y_{k-1})$

This is the distribution of the measurement *before* we actually observe $y_k^{obs}$, conditioned only on all previous measurements $Y_{k-1}$.
Given $Y_{k-1}$, our state belief is $x_k^- \sim \mathcal{N}(m_k^-, P_k^-)$.

**Expected Value:**
$$E[y_k^- \mid Y_{k-1}] = E[h x_k^- + z_k] = h E[x_k^-] + E[z_k] = h m_k^-$$

**Variance:**
$$Var(y_k^- \mid Y_{k-1}) = Var(h x_k^- + z_k) = h^2 Var(x_k^-) + Var(z_k) = h^2 P_k^- + r$$

Therefore:
$$\mathbf{p(y_k^- \mid Y_{k-1}) = \mathcal{N}(h m_k^-, h^2 P_k^- + r)}$$

---

#### 4. Show the Posterior-Predictive Measurement Distribution $p(y_k^- \mid Y_k)$

This represents our belief about what the measurement *should* be, *after* we have already incorporated $y_k^{obs}$ into our state estimate.
Given $Y_k$, our updated state belief is $x_k^+ \sim \mathcal{N}(m_k, P_k)$.

**Expected Value:**
$$E[y_k \mid Y_k] = E[h x_k^+ + z_k] = h E[x_k^+] + E[z_k] = h m_k$$

**Variance:**
$$Var(y_k \mid Y_k) = Var(h x_k^+ + z_k) = h^2 Var(x_k^+) + Var(z_k) = h^2 P_k + r$$

Therefore:
$$\mathbf{p(y_k^- \mid Y_k) = \mathcal{N}(h m_k, h^2 P_k + r)}$$

# 2D Constant-Velocity Target Tracking: Matrix Derivations

## Part A

### 2D Constant-Velocity Target Tracking: Matrix Derivations

**Given System State:**
The hidden state vector $x_k$ at time step $k$ contains the 2D position and 2D velocity:
$$x_k = \begin{bmatrix} p_x(k) \\ p_y(k) \\ v_x(k) \\ v_y(k) \end{bmatrix}$$

**Given Measurement:**
The measurement vector $y_k$ contains only the noisy observations of the 2D position:
$$y_k = \begin{bmatrix} p_x^{meas}(k) \\ p_y^{meas}(k) \end{bmatrix}$$

---

#### 1. Derivation of the State Transition Matrix ($A$)

The state transition matrix $A$ dictates how the state evolves over a sampling interval $\Delta t$ in the absence of noise. For a constant-velocity model, we assume the acceleration is zero.

From basic kinematics:
* **Position** changes based on the previous position and velocity: $p = p_{old} + v_{old}\Delta t$
* **Velocity** remains constant: $v = v_{old}$

Applying these kinematic equations to our specific $x$ and $y$ coordinates:
1.  $p_x(k) = p_x(k-1) + \Delta t \, v_x(k-1)$
2.  $p_y(k) = p_y(k-1) + \Delta t \, v_y(k-1)$
3.  $v_x(k) = v_x(k-1)$
4.  $v_y(k) = v_y(k-1)$

To express this as a matrix multiplication $x_k = A x_{k-1}$, we map the coefficients of $x_{k-1}$ into the rows of $A$:

$$
\begin{bmatrix} p_x(k) \\ p_y(k) \\ v_x(k) \\ v_y(k) \end{bmatrix}
=
\begin{bmatrix}
1 & 0 & \Delta t & 0 \\
0 & 1 & 0 & \Delta t \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix} p_x(k-1) \\ p_y(k-1) \\ v_x(k-1) \\ v_y(k-1) \end{bmatrix}
$$

Therefore:
$$\mathbf{A = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}}$$

---

#### 2. Derivation of the Measurement Matrix ($H$)

The measurement matrix $H$ maps the true state space into the measurement space.
We only have sensors measuring the position ($p_x, p_y$), so the velocity components ($v_x, v_y$) must be ignored (multiplied by zero).

The measurement equation is $y_k = H x_k + z_k$. Ignoring the noise $z_k$ for the mapping:

$$
\begin{bmatrix} p_x^{meas}(k) \\ p_y^{meas}(k) \end{bmatrix}
=
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0
\end{bmatrix}
\begin{bmatrix} p_x(k) \\ p_y(k) \\ v_x(k) \\ v_y(k) \end{bmatrix}
$$

Therefore:
$$\mathbf{H = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}}$$

---

#### 3. Derivation of the Process Noise Mapping Matrix ($G$)

In a constant-velocity model, we often model the "unmodeled dynamics" (like a sudden gust of wind or a pilot maneuver) as random, white acceleration noise $w_{k-1}$ that occurs between time steps.

Let the 2D acceleration noise be $w_{k-1} = \begin{bmatrix} a_x \\ a_y \end{bmatrix}$.

From basic kinematics, a constant acceleration $a$ applied over a time interval $\Delta t$ affects both velocity and position:
* $\Delta p = \frac{1}{2} a \Delta t^2$
* $\Delta v = a \Delta t$

Adding these noise components to our exact kinematic equations from Part 1:
1.  $p_x(k) = p_x(k-1) + \Delta t \, v_x(k-1) + \mathbf{\frac{1}{2} \Delta t^2 a_x}$
2.  $p_y(k) = p_y(k-1) + \Delta t \, v_y(k-1) + \mathbf{\frac{1}{2} \Delta t^2 a_y}$
3.  $v_x(k) = v_x(k-1) + \mathbf{\Delta t \, a_x}$
4.  $v_y(k) = v_y(k-1) + \mathbf{\Delta t \, a_y}$

To express the noise injection as $G w_{k-1}$, we map the coefficients of $a_x$ and $a_y$ into the matrix $G$:

$$
G w_{k-1} =
\begin{bmatrix}
\frac{1}{2} \Delta t^2 & 0 \\
0 & \frac{1}{2} \Delta t^2 \\
\Delta t & 0 \\
0 & \Delta t
\end{bmatrix}
\begin{bmatrix} a_x \\ a_y \end{bmatrix}
$$

Therefore:
$$\mathbf{G = \begin{bmatrix} \frac{1}{2} \Delta t^2 & 0 \\ 0 & \frac{1}{2} \Delta t^2 \\ \Delta t & 0 \\ 0 & \Delta t \end{bmatrix}}$$

## Part B

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Define the Kalman Filter Class
# ==========================================
class ConstantVelocityKalmanFilter:
    def __init__(self, dt, q_variance, r_variance, initial_state, initial_covariance):
        self.dt = dt

        # State Transition Matrix (A) from Part A
        self.A = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])

        # Measurement Matrix (H) from Part A
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])

        # Process Noise Covariance (Q = G * Sigma_p * G^T)
        # Using the G matrix derived in Part A
        G = np.array([
            [0.5 * dt**2, 0],
            [0, 0.5 * dt**2],
            [dt, 0],
            [0, dt]
        ])
        self.Q = q_variance * (G @ G.T)

        # Measurement Noise Covariance (R)
        self.R = np.array([
            [r_variance, 0],
            [0, r_variance]
        ])

        # Initialize State (x) and Covariance (P)
        self.x = np.array(initial_state, dtype=float).reshape(4, 1)
        self.P = np.array(initial_covariance, dtype=float)

    def predict(self):
        # Time Update: x_k^- = A * x_{k-1}^+
        self.x = self.A @ self.x

        # Covariance Update: P_k^- = A * P_{k-1}^+ * A^T + Q
        self.P = self.A @ self.P @ self.A.T + self.Q
        return self.x

    def update(self, z):
        # Format measurement as column vector
        z = np.array(z, dtype=float).reshape(2, 1)

        # Innovation (Residual): y = z - H * x^-
        y = z - self.H @ self.x

        # Innovation Covariance: S = H * P^- * H^T + R
        S = self.H @ self.P @ self.H.T + self.R

        # Kalman Gain: K = P^- * H^T * S^-1
        K = self.P @ self.H.T @ np.linalg.inv(S)

        # Posterior State Update: x^+ = x^- + K * y
        self.x = self.x + K @ y

        # Posterior Covariance Update: P^+ = (I - K * H) * P^-
        I = np.eye(self.P.shape[0])
        self.P = (I - K @ self.H) @ self.P

        return self.x

# ==========================================
# 2. Simulate Noisy GPS Data
# ==========================================
np.random.seed(42) # For reproducible results
dt = 1.0           # 1 second sampling interval
num_steps = 60

true_positions = []
noisy_measurements = []

# Ground truth initial conditions
true_x, true_y = 0.0, 0.0
true_vx, true_vy = 10.0, 5.0 # Target is moving diagonally

# Define the standard deviation of our GPS noise (e.g., 20 meters)
gps_noise_std = 20.0

for _ in range(num_steps):
    # Simulate true kinematics
    true_x += true_vx * dt
    true_y += true_vy * dt
    true_positions.append((true_x, true_y))

    # Simulate noisy GPS readings
    meas_x = true_x + np.random.normal(0, gps_noise_std)
    meas_y = true_y + np.random.normal(0, gps_noise_std)
    noisy_measurements.append((meas_x, meas_y))

# ==========================================
# 3. Run the Kalman Filter
# ==========================================
# We initialize the filter blind (starting at [0,0] with zero velocity)
# We set P0 very high because we have no idea if our initial guess is right.
initial_state = [0, 0, 0, 0]
initial_covariance = np.eye(4) * 1000

kf = ConstantVelocityKalmanFilter(
    dt=dt,
    q_variance=0.5,             # Tuning parameter: expected random acceleration
    r_variance=gps_noise_std**2, # Sensor variance is std_dev squared
    initial_state=initial_state,
    initial_covariance=initial_covariance
)

filtered_positions = []

# Loop through the incoming GPS stream
for z in noisy_measurements:
    kf.predict()                 # Step 1: Predict based on velocity
    state = kf.update(z)         # Step 2: Correct using the GPS ping

    # Save the filtered [x, y] position
    filtered_positions.append((state[0, 0], state[1, 0]))

# ==========================================
# 4. Visualize the Results
# ==========================================
true_x_vals, true_y_vals = zip(*true_positions)
meas_x_vals, meas_y_vals = zip(*noisy_measurements)
filt_x_vals, filt_y_vals = zip(*filtered_positions)

plt.figure(figsize=(10, 8))
plt.plot(true_x_vals, true_y_vals, 'g--', label='True Trajectory', linewidth=2)
plt.scatter(meas_x_vals, meas_y_vals, c='r', marker='x', label='Noisy GPS Measurements', alpha=0.6)
plt.plot(filt_x_vals, filt_y_vals, 'b-', label='Kalman Filter Estimate', linewidth=2.5)

plt.title("2D Constant-Velocity Kalman Filter on Noisy GPS Data", fontsize=14)
plt.xlabel("X Position (meters)")
plt.ylabel("Y Position (meters)")
plt.legend()
plt.grid(True)
plt.show()